# Survival Model Comparison for Major Comment 2 Response

**Purpose:** Compare M-CARD AI (binary XGBoost classifier) against same-cohort, same-feature survival estimators:
- Cox proportional hazards
- Random Survival Forest
- DeepSurv
- XGBoost with `survival:cox` objective

**Evaluation metrics on test set (Scotland/Wales, n=3,881):**
- Harrell's C-index
- Time-dependent AUC at 1–9 years (IPCW)
- Mean TD-AUC across 1–9 years
- Brier score at 10 years (IPCW)

**All metrics reported with 1,000 bootstrap 95% CIs.**

---

## Important assumptions on input

This notebook expects the user to have already produced the following variables from their preprocessing pipeline (`train_test_comfort` + `create_survival_vars`):

| Variable | Shape | Description |
|---|---|---|
| `X_train_raw` | (n_train, 26) | **Imputed + scaled** training features, **BEFORE ADASYN** |
| `y_train_raw` | (n_train,) | Binary outcome for training, **BEFORE ADASYN** |
| `X_val` | (n_val, 26) | Imputed + scaled validation features |
| `X_external` | (n_test, 26) | Imputed + scaled test features (Scotland/Wales) |
| `time_train` | (n_train,) | Time to event/censoring in **years**, aligned with `X_train_raw` |
| `event_train` | (n_train,) | 1=CVD event, 0=censored, aligned with `X_train_raw` |
| `time_val`, `event_val` | (n_val,) | Same as above, for validation set |
| `time_ext`, `event_ext` | (n_test,) | Same as above, for test set |

**Critical:** Survival models must be trained on data BEFORE ADASYN resampling. ADASYN duplicates/synthesizes minority-class samples in a way that destroys the integrity of time-to-event information (synthetic samples have no real follow-up time). If your `train_test_comfort` returns `X_train, y_train` after ADASYN, you need to also return the pre-ADASYN versions for use here.

If your current pipeline only returns post-ADASYN `X_train, y_train`, modify it to also return `X_train_raw, y_train_raw` (i.e., the imputed + scaled features without ADASYN applied).

## Required libraries

```bash
pip install scikit-survival lifelines pycox torch torchtuples xgboost
```


## 0. Imports and configuration

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Survival modeling
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc, brier_score

# XGBoost survival
import xgboost as xgb

# DeepSurv
import torch
import torchtuples as tt
from pycox.models import CoxPH as DeepSurv

# Reproducibility
RANDOM_STATE = 777
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Time points for time-dependent AUC (1–9 years, matching manuscript)
TD_AUC_TIMES = np.arange(1, 10).astype(float)

# Brier score evaluation horizon
BRIER_HORIZON = 10.0

# Bootstrap configuration
N_BOOTSTRAP = 1000

print(f"Random state: {RANDOM_STATE}")
print(f"TD-AUC times (years): {TD_AUC_TIMES}")
print(f"Brier horizon (years): {BRIER_HORIZON}")
print(f"Bootstrap resamples: {N_BOOTSTRAP}")

## 1. Input data check

Verify all required inputs exist with consistent shapes.

In [ ]:
# Verify required arrays exist (will raise NameError if not defined)
required_vars = [
    'X_train_raw', 'y_train_raw',
    'X_val', 'y_val',
    'X_external',
    'time_train', 'event_train',
    'time_val', 'event_val',
    'time_ext', 'event_ext'
]
for v in required_vars:
    assert v in dir(), f"Required variable `{v}` is not defined. Run preprocessing first."

# Convert to numpy arrays if DataFrames
def _to_np(x):
    return x.values if hasattr(x, 'values') else np.asarray(x)

X_train_np = _to_np(X_train_raw).astype(float)
X_val_np = _to_np(X_val).astype(float)
X_test_np = _to_np(X_external).astype(float)

time_train_np = _to_np(time_train).astype(float)
event_train_np = _to_np(event_train).astype(bool)
time_val_np = _to_np(time_val).astype(float)
event_val_np = _to_np(event_val).astype(bool)
time_test_np = _to_np(time_ext).astype(float)
event_test_np = _to_np(event_ext).astype(bool)

# Shape checks
assert X_train_np.shape[0] == len(time_train_np) == len(event_train_np)
assert X_val_np.shape[0] == len(time_val_np) == len(event_val_np)
assert X_test_np.shape[0] == len(time_test_np) == len(event_test_np)
assert X_train_np.shape[1] == X_val_np.shape[1] == X_test_np.shape[1]

print(f"Train: X={X_train_np.shape}, events={event_train_np.sum()}/{len(event_train_np)} ({100*event_train_np.mean():.2f}%)")
print(f"Val:   X={X_val_np.shape}, events={event_val_np.sum()}/{len(event_val_np)} ({100*event_val_np.mean():.2f}%)")
print(f"Test:  X={X_test_np.shape}, events={event_test_np.sum()}/{len(event_test_np)} ({100*event_test_np.mean():.2f}%)")
print(f"\nTime range (years): train [{time_train_np.min():.2f}, {time_train_np.max():.2f}]")
print(f"                    val   [{time_val_np.min():.2f}, {time_val_np.max():.2f}]")
print(f"                    test  [{time_test_np.min():.2f}, {time_test_np.max():.2f}]")

# Build structured arrays for scikit-survival
y_surv_train = Surv.from_arrays(event=event_train_np, time=time_train_np)
y_surv_val = Surv.from_arrays(event=event_val_np, time=time_val_np)
y_surv_test = Surv.from_arrays(event=event_test_np, time=time_test_np)

## 2. Train survival models

All models trained on the same training set (England, n≈28,648), with no ADASYN.

Each model produces a **risk score** on the test set, which is used for:
- C-index (higher risk → earlier event)
- Time-dependent AUC at year `t`: requires `P(T ≤ t | X)` or proportional risk score

For metrics that require survival function `S(t|X)` (Brier score at 10 years), we use each model's native survival function output.


### 2.1 Cox proportional hazards (scikit-survival)

In [ ]:
cox_model = CoxPHSurvivalAnalysis(alpha=1e-4)  # small ridge for numerical stability
cox_model.fit(X_train_np, y_surv_train)

# Risk score on test set (linear predictor)
cox_risk_test = cox_model.predict(X_test_np)

# Survival function at horizon for Brier score
cox_surv_funcs_test = cox_model.predict_survival_function(X_test_np)
cox_S_at_horizon_test = np.array([sf(BRIER_HORIZON) for sf in cox_surv_funcs_test])

print(f"Cox: trained. Test risk score range [{cox_risk_test.min():.3f}, {cox_risk_test.max():.3f}]")

### 2.2 Random Survival Forest

In [ ]:
rsf_model = RandomSurvivalForest(
    n_estimators=500,
    min_samples_split=10,
    min_samples_leaf=15,
    max_features='sqrt',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rsf_model.fit(X_train_np, y_surv_train)

# Risk score: predict() returns ensemble mortality (higher = higher risk)
rsf_risk_test = rsf_model.predict(X_test_np)

# Survival function at horizon
rsf_surv_funcs_test = rsf_model.predict_survival_function(X_test_np)
rsf_S_at_horizon_test = np.array([sf(BRIER_HORIZON) for sf in rsf_surv_funcs_test])

print(f"RSF: trained. Test risk score range [{rsf_risk_test.min():.3f}, {rsf_risk_test.max():.3f}]")

### 2.3 XGBoost survival (Cox objective)

In [ ]:
# XGBoost survival:cox expects labels: positive = event time, negative = censoring time
def make_xgb_surv_label(time, event):
    label = np.where(event, time, -time)
    return label

y_xgb_train = make_xgb_surv_label(time_train_np, event_train_np)
y_xgb_val = make_xgb_surv_label(time_val_np, event_val_np)

dtrain = xgb.DMatrix(X_train_np, label=y_xgb_train)
dval = xgb.DMatrix(X_val_np, label=y_xgb_val)
dtest = xgb.DMatrix(X_test_np)

xgb_params = {
    'objective': 'survival:cox',
    'eval_metric': 'cox-nloglik',
    'learning_rate': 0.05,
    'max_depth': 4,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'seed': RANDOM_STATE,
    'verbosity': 0,
}

xgb_surv_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=2000,
    evals=[(dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=False,
)

# Output is exp(linear predictor) — a hazard ratio. Higher = higher risk.
xgb_risk_test = xgb_surv_model.predict(dtest)

# For Brier score we need S(10|X). XGBoost survival:cox does not return baseline hazard;
# we estimate it from training data using Breslow's estimator.
xgb_train_risk = xgb_surv_model.predict(dtrain)

def breslow_baseline_survival(train_time, train_event, train_risk, eval_times):
    """Breslow estimator of baseline cumulative hazard, then S0(t) = exp(-H0(t))."""
    order = np.argsort(train_time)
    t_sorted = train_time[order]
    e_sorted = train_event[order]
    r_sorted = train_risk[order]
    # cumulative sum of risks for at-risk set (from the end)
    risk_sum_at_risk = np.cumsum(r_sorted[::-1])[::-1]
    # baseline hazard increment at each event time
    H0_eval = np.zeros_like(eval_times)
    for i, t in enumerate(eval_times):
        mask = (t_sorted <= t) & e_sorted
        if not mask.any():
            H0_eval[i] = 0.0
        else:
            # dH0 at each event time = 1 / sum(risk of those at risk)
            dH0 = 1.0 / risk_sum_at_risk[mask]
            H0_eval[i] = dH0.sum()
    S0_eval = np.exp(-H0_eval)
    return S0_eval

S0_at_horizon = breslow_baseline_survival(
    time_train_np, event_train_np.astype(int), xgb_train_risk, np.array([BRIER_HORIZON])
)[0]
xgb_S_at_horizon_test = S0_at_horizon ** xgb_risk_test

print(f"XGB-Surv: trained. Best iter={xgb_surv_model.best_iteration}")
print(f"          Test risk score range [{xgb_risk_test.min():.3f}, {xgb_risk_test.max():.3f}]")
print(f"          Baseline S0(10y)={S0_at_horizon:.4f}")

### 2.4 DeepSurv (pycox)

Single hidden-layer MLP optimized with the Cox partial likelihood.

In [ ]:
in_features = X_train_np.shape[1]
num_nodes = [64, 32]
out_features = 1
batch_norm = True
dropout = 0.3

net = tt.practical.MLPVanilla(
    in_features=in_features,
    num_nodes=num_nodes,
    out_features=out_features,
    batch_norm=batch_norm,
    dropout=dropout,
    output_bias=False,
)

deepsurv_model = DeepSurv(net, tt.optim.Adam(lr=1e-3))

# pycox label format: (time_array, event_array)
y_ds_train = (time_train_np.astype('float32'), event_train_np.astype('float32'))
y_ds_val = (time_val_np.astype('float32'), event_val_np.astype('float32'))

X_train_ds = X_train_np.astype('float32')
X_val_ds = X_val_np.astype('float32')
X_test_ds = X_test_np.astype('float32')

log = deepsurv_model.fit(
    X_train_ds, y_ds_train,
    batch_size=256,
    epochs=100,
    callbacks=[tt.callbacks.EarlyStopping(patience=10)],
    val_data=(X_val_ds, y_ds_val),
    verbose=False,
)

# Compute baseline hazard from training data (required for survival function)
_ = deepsurv_model.compute_baseline_hazards()

# Risk score = network output (log hazard ratio)
deepsurv_risk_test = deepsurv_model.predict(X_test_ds).flatten()

# Survival function at horizon
surv_df = deepsurv_model.predict_surv_df(X_test_ds)  # rows: time, cols: patients
# Interpolate to BRIER_HORIZON
available_times = surv_df.index.values.astype(float)
if BRIER_HORIZON in available_times:
    deepsurv_S_at_horizon_test = surv_df.loc[BRIER_HORIZON].values
else:
    # find closest <= horizon
    valid = available_times[available_times <= BRIER_HORIZON]
    if len(valid) > 0:
        t_use = valid.max()
        deepsurv_S_at_horizon_test = surv_df.loc[t_use].values
    else:
        deepsurv_S_at_horizon_test = np.ones(len(X_test_np))

print(f"DeepSurv: trained. Final epoch={len(log.to_pandas())}")
print(f"          Test risk score range [{deepsurv_risk_test.min():.3f}, {deepsurv_risk_test.max():.3f}]")

## 3. Performance evaluation on test set

Three metrics:
1. **Harrell's C-index** — `concordance_index_censored` (no IPCW; pairwise on orderable pairs)
2. **Time-dependent AUC at 1–9 years** — `cumulative_dynamic_auc` (IPCW)
3. **Brier score at 10 years** — `brier_score` (IPCW)

For each metric, 1,000 bootstrap resamples of the test set produce 95% CIs.

In [ ]:
# Risk scores from each survival model (higher = higher 10-year risk)
risk_scores_test = {
    'Cox PH': cox_risk_test,
    'RSF': rsf_risk_test,
    'XGBoost-Surv': xgb_risk_test,
    'DeepSurv': deepsurv_risk_test,
}

# Survival probabilities at 10 years (for Brier score; lower S(10) = higher risk)
S10_test = {
    'Cox PH': cox_S_at_horizon_test,
    'RSF': rsf_S_at_horizon_test,
    'XGBoost-Surv': xgb_S_at_horizon_test,
    'DeepSurv': deepsurv_S_at_horizon_test,
}

# Sanity check: each must be 1D array of length n_test
for name, rs in risk_scores_test.items():
    assert rs.shape == (len(time_test_np),), f"{name} risk score shape mismatch: {rs.shape}"
    assert S10_test[name].shape == (len(time_test_np),), f"{name} S(10) shape mismatch"
    print(f"{name}: risk score mean={rs.mean():.3f}, S(10) mean={S10_test[name].mean():.3f}")

### 3.1 Point estimates

In [ ]:
def compute_metrics_point(y_train_surv, y_test_surv, risk_test, S10_test_arr, td_times, brier_horizon):
    """Compute point estimates of C-index, TD-AUCs, and Brier score on test set.
    Censoring weights for TD-AUC and Brier score are estimated from training data.
    """
    # C-index (no IPCW)
    cidx_result = concordance_index_censored(
        y_test_surv['event'], y_test_surv['time'], risk_test
    )
    c_index = cidx_result[0]

    # Time-dependent AUC (IPCW; censoring distribution from training set)
    # cumulative_dynamic_auc returns (auc_per_time, mean_auc)
    try:
        td_auc_vals, td_auc_mean = cumulative_dynamic_auc(
            y_train_surv, y_test_surv, risk_test, td_times
        )
    except Exception as e:
        td_auc_vals = np.full(len(td_times), np.nan)
        td_auc_mean = np.nan

    # Brier score at 10 years (IPCW)
    try:
        # brier_score needs survival probability at the eval time
        times_eval, bs_vals = brier_score(
            y_train_surv, y_test_surv, S10_test_arr, np.array([brier_horizon])
        )
        brier = bs_vals[0]
    except Exception as e:
        brier = np.nan

    return c_index, td_auc_vals, td_auc_mean, brier


point_results = {}
for name in risk_scores_test:
    c_idx, td_vals, td_mean, bs = compute_metrics_point(
        y_surv_train, y_surv_test, risk_scores_test[name], S10_test[name],
        TD_AUC_TIMES, BRIER_HORIZON
    )
    point_results[name] = {
        'c_index': c_idx,
        'td_auc_per_year': td_vals,
        'td_auc_mean': td_mean,
        'brier_10y': bs,
    }
    print(f"\n{name}")
    print(f"  C-index:      {c_idx:.4f}")
    print(f"  Mean TD-AUC:  {td_mean:.4f}")
    for yr, v in zip(TD_AUC_TIMES, td_vals):
        print(f"    Year {int(yr)}:    {v:.4f}")
    print(f"  Brier (10y):  {bs:.4f}")

### 3.2 Bootstrap 95% confidence intervals

1,000 resamples of the test set. Same random seed across models for paired comparison.

In [ ]:
def bootstrap_ci(y_train_surv, y_test_surv, risk_test, S10_arr, td_times, brier_horizon,
                 n_boot=1000, seed=777):
    rng = np.random.default_rng(seed)
    n_test = len(y_test_surv)

    cidx_boot = []
    td_per_year_boot = []
    td_mean_boot = []
    brier_boot = []

    for b in range(n_boot):
        idx = rng.integers(0, n_test, size=n_test)
        y_test_b = y_test_surv[idx]
        risk_b = risk_test[idx]
        S10_b = S10_arr[idx]

        # Skip resamples with no events (metrics undefined)
        if y_test_b['event'].sum() < 2:
            continue

        try:
            c, _ = concordance_index_censored(
                y_test_b['event'], y_test_b['time'], risk_b
            )[:2]
        except Exception:
            c = np.nan
        cidx_boot.append(c)

        try:
            td_vals_b, td_mean_b = cumulative_dynamic_auc(
                y_train_surv, y_test_b, risk_b, td_times
            )
        except Exception:
            td_vals_b = np.full(len(td_times), np.nan)
            td_mean_b = np.nan
        td_per_year_boot.append(td_vals_b)
        td_mean_boot.append(td_mean_b)

        try:
            _, bs_vals = brier_score(
                y_train_surv, y_test_b, S10_b, np.array([brier_horizon])
            )
            brier_boot.append(bs_vals[0])
        except Exception:
            brier_boot.append(np.nan)

    cidx_arr = np.array(cidx_boot)
    td_per_year_arr = np.array(td_per_year_boot)
    td_mean_arr = np.array(td_mean_boot)
    brier_arr = np.array(brier_boot)

    def pct_ci(a):
        a = a[~np.isnan(a)]
        if len(a) == 0:
            return (np.nan, np.nan)
        return (np.percentile(a, 2.5), np.percentile(a, 97.5))

    return {
        'c_index_ci': pct_ci(cidx_arr),
        'td_auc_mean_ci': pct_ci(td_mean_arr),
        'td_auc_per_year_ci': [pct_ci(td_per_year_arr[:, i]) for i in range(td_per_year_arr.shape[1])],
        'brier_ci': pct_ci(brier_arr),
        '_boot_arrays': {
            'c_index': cidx_arr,
            'td_mean': td_mean_arr,
            'td_per_year': td_per_year_arr,
            'brier': brier_arr,
        }
    }


boot_results = {}
for name in risk_scores_test:
    print(f"Bootstrap for {name}...")
    boot_results[name] = bootstrap_ci(
        y_surv_train, y_surv_test,
        risk_scores_test[name], S10_test[name],
        TD_AUC_TIMES, BRIER_HORIZON,
        n_boot=N_BOOTSTRAP, seed=RANDOM_STATE
    )
    print(f"  C-index 95% CI:   ({boot_results[name]['c_index_ci'][0]:.4f}, {boot_results[name]['c_index_ci'][1]:.4f})")
    print(f"  Mean TD-AUC 95%:  ({boot_results[name]['td_auc_mean_ci'][0]:.4f}, {boot_results[name]['td_auc_mean_ci'][1]:.4f})")
    print(f"  Brier 10y 95%:    ({boot_results[name]['brier_ci'][0]:.4f}, {boot_results[name]['brier_ci'][1]:.4f})")

## 4. Summary table

Format matching manuscript Supplementary Table style (e.g., Table S2 / S9).

In [ ]:
def fmt(point, ci):
    return f"{point:.3f} ({ci[0]:.3f}–{ci[1]:.3f})"

rows = []
for name in risk_scores_test:
    p = point_results[name]
    b = boot_results[name]
    row = {
        'Model': name,
        'C-index (95% CI)': fmt(p['c_index'], b['c_index_ci']),
        'Mean TD-AUC (95% CI)': fmt(p['td_auc_mean'], b['td_auc_mean_ci']),
        'Brier at 10y (95% CI)': fmt(p['brier_10y'], b['brier_ci']),
    }
    for yr_idx, yr in enumerate(TD_AUC_TIMES):
        row[f'TD-AUC y{int(yr)}'] = fmt(p['td_auc_per_year'][yr_idx], b['td_auc_per_year_ci'][yr_idx])
    rows.append(row)

summary_df = pd.DataFrame(rows)
print("\n=== Summary table ===")
with pd.option_context('display.max_columns', None, 'display.width', 200):
    print(summary_df.to_string(index=False))

In [ ]:
# Save outputs
summary_df.to_csv('survival_model_comparison_summary.csv', index=False)
print("Summary saved to: survival_model_comparison_summary.csv")

# Save raw bootstrap arrays for any follow-up analysis (e.g., DeLong-style comparison)
boot_arrays_to_save = {
    name: {
        'c_index': boot_results[name]['_boot_arrays']['c_index'],
        'td_mean': boot_results[name]['_boot_arrays']['td_mean'],
        'brier': boot_results[name]['_boot_arrays']['brier'],
    }
    for name in risk_scores_test
}
np.savez('survival_model_bootstrap_arrays.npz', **{
    f"{name}__{metric}": arr
    for name, d in boot_arrays_to_save.items()
    for metric, arr in d.items()
})
print("Bootstrap arrays saved to: survival_model_bootstrap_arrays.npz")

## 5. Notes for integration into manuscript / response letter

**For the response letter (Major Comment 2, Section 4d):**
- The M-CARD AI test-set C-index 0.752 (0.722–0.781) and mean TD-AUC 0.785 (0.754–0.825) are already in Table 3 / Supplementary Table S2 of the manuscript.
- Compare against the same-cohort survival estimators above; report results honestly regardless of direction.

**For the supplementary materials (new Supplementary Table S10):**
Suggested structure: rows = {M-CARD AI, Cox PH, RSF, XGBoost-Surv, DeepSurv}, columns = {C-index, Mean TD-AUC, Brier at 10y}. M-CARD AI row uses pre-existing manuscript values; other four rows use the values computed in this notebook.

**Honest reporting reminder:**
If any survival estimator achieves a C-index or mean TD-AUC with a CI that excludes the M-CARD AI point estimate, the response letter must reflect this honestly rather than claim uniform M-CARD AI superiority. The empirical comparison is the substance of the response — the conclusion follows from the numbers, not the other way around.
